# Unsupervised Tribute - Art Texture Generator

Generate fluid, art-inspired texture atlases for the Unsupervised visualization.

**Inspired by Refik Anadol's "Unsupervised" at MoMA**

## Dataset:
- Best Artworks of All Time (~8,000 images)

## To Expand:
Add images to `images/` directory and re-run from Section 4.

---
## 1. Setup

In [ ]:
# Install dependencies (use Colab's PyTorch to avoid conflicts)
!pip install -q diffusers transformers accelerate kagglehub scipy tqdm

import torch
import numpy as np
from PIL import Image
import os
import shutil
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
import hashlib
import kagglehub

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Configuration
CONFIG = {
    "images_dir": "./images",
    "output_dir": "./output",
    "cache_dir": "./cache",
    "atlas_size": 1024,
    "grid_size": 4,
    "num_atlases": 3,
    "max_images": 2000,
    "batch_size": 4,
    "blur_sigma": 2.5,
    "image_size": 512,
}

for d in ["images_dir", "output_dir", "cache_dir"]:
    Path(CONFIG[d]).mkdir(parents=True, exist_ok=True)

print("Config:", CONFIG)

---
## 2. Download Art Dataset (via kagglehub)

In [ ]:
# Download Best Artworks dataset directly via kagglehub
print("Downloading Best Artworks dataset...")
artworks_path = kagglehub.dataset_download("ikarus777/best-artworks-of-all-time")
print(f"Downloaded to: {artworks_path}")

# Show what's in the downloaded directory
artworks_dir = Path(artworks_path)
print(f"\nContents of {artworks_path}:")
for item in list(artworks_dir.iterdir())[:10]:
    print(f"  {item.name}")
if len(list(artworks_dir.iterdir())) > 10:
    print(f"  ... and {len(list(artworks_dir.iterdir())) - 10} more items")

In [ ]:
# Consolidate images from downloaded dataset
images_dir = Path(CONFIG["images_dir"])
image_exts = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def collect_images(source_path, dest_dir):
    """Collect images from source directory to destination."""
    count = 0
    source = Path(source_path)
    
    # Find all image files recursively
    for ext in image_exts:
        for img in source.rglob(f"*{ext}"):
            try:
                h = hashlib.md5(str(img).encode()).hexdigest()[:8]
                dest = dest_dir / f"{img.stem}_{h}{img.suffix.lower()}"
                if not dest.exists():
                    shutil.copy2(img, dest)
                    count += 1
            except Exception as e:
                pass  # Skip problematic files
        for img in source.rglob(f"*{ext.upper()}"):
            try:
                h = hashlib.md5(str(img).encode()).hexdigest()[:8]
                dest = dest_dir / f"{img.stem}_{h}{img.suffix.lower()}"
                if not dest.exists():
                    shutil.copy2(img, dest)
                    count += 1
            except Exception as e:
                pass
    return count

print("Collecting images from Best Artworks...")
count = collect_images(artworks_path, images_dir)
print(f"Added {count} images")

total = len([p for p in images_dir.glob("*") if p.suffix.lower() in image_exts])
print(f"\nTotal images in images/: {total}")

if total == 0:
    print("\nWARNING: No images found! Let's check the dataset structure...")
    for item in Path(artworks_path).rglob("*"):
        if item.is_file():
            print(f"  Found file: {item}")
            if len(list(Path(artworks_path).rglob("*"))) > 20:
                print("  ... (showing first 20)")
                break

---
## 3. Load VAE

In [ ]:
from diffusers import AutoencoderKL
from torchvision import transforms

print("Loading VAE...")
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
    torch_dtype=torch.float16
).to(device)
vae.eval()
print("VAE loaded!")

preprocess = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.CenterCrop(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

---
## 4. Encode Images (re-run from here when adding images)

In [ ]:
# Scan images
images_dir = Path(CONFIG["images_dir"])
image_exts = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
all_paths = sorted(set(
    list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.jpeg")) +
    list(images_dir.glob("*.png")) + list(images_dir.glob("*.webp")) +
    list(images_dir.glob("*.bmp")) + list(images_dir.glob("*.JPG")) +
    list(images_dir.glob("*.JPEG")) + list(images_dir.glob("*.PNG"))
))
print(f"Found {len(all_paths)} images")

if len(all_paths) == 0:
    raise ValueError("No images found! Please check Section 2 output and ensure images were downloaded.")

# Show samples
samples = np.random.choice(all_paths, min(8, len(all_paths)), replace=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.flat, samples):
    try:
        ax.imshow(Image.open(p).convert('RGB'))
        ax.set_title(p.stem[:20], fontsize=8)
    except: pass
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Caching helpers
cache_dir = Path(CONFIG["cache_dir"])

def cache_path(p):
    s = p.stat()
    return cache_dir / f"{hashlib.md5(f'{p}_{s.st_mtime}_{s.st_size}'.encode()).hexdigest()}.npy"

@torch.no_grad()
def encode_batch(paths):
    imgs, valid = [], []
    for p in paths:
        try:
            imgs.append(preprocess(Image.open(p).convert('RGB')))
            valid.append(p)
        except Exception as e:
            print(f"Skipping {p.name}: {e}")
    if not imgs:
        return [], []
    batch = torch.stack(imgs).to(device, dtype=torch.float16)
    return vae.encode(batch).latent_dist.sample().cpu().numpy(), valid

In [ ]:
# Encode with caching
max_imgs = CONFIG["max_images"]
batch_size = CONFIG["batch_size"]

selected = list(np.random.choice(all_paths, min(max_imgs, len(all_paths)), replace=False)) if len(all_paths) > max_imgs else list(all_paths)
print(f"Processing {len(selected)} images")

cached, uncached = [], []
for p in tqdm(selected, desc="Cache check"):
    cp = cache_path(p)
    if cp.exists():
        try:
            cached.append(np.load(cp))
        except:
            uncached.append(p)
    else:
        uncached.append(p)

print(f"Cached: {len(cached)}, To encode: {len(uncached)}")

new_latents = []
for i in tqdm(range(0, len(uncached), batch_size), desc="Encoding"):
    lats, paths = encode_batch(uncached[i:i+batch_size])
    for lat, p in zip(lats, paths):
        l = lat[np.newaxis, ...]
        np.save(cache_path(p), l)
        new_latents.append(l)

all_latents = cached + new_latents
if not all_latents:
    raise ValueError("No images were successfully encoded! Check for errors above.")

latents = np.concatenate(all_latents, axis=0)
print(f"Total latents: {len(latents)}")

---
## 5. Generate Atlases

In [ ]:
# Interpolation
def slerp(v0, v1, t):
    v0f, v1f = v0.flatten(), v1.flatten()
    n0, n1 = np.linalg.norm(v0f), np.linalg.norm(v1f)
    if n0 < 1e-8 or n1 < 1e-8:
        return v0*(1-t) + v1*t
    dot = np.clip(np.dot(v0f/n0, v1f/n1), -1, 1)
    theta = np.arccos(dot)
    if abs(theta) < 1e-6:
        return v0*(1-t) + v1*t
    sin_theta = np.sin(theta)
    if abs(sin_theta) < 1e-8:
        return v0*(1-t) + v1*t
    return (np.sin((1-t)*theta)/sin_theta*v0f + np.sin(t*theta)/sin_theta*v1f).reshape(v0.shape)

def gen_path(latents, n, seed=42):
    np.random.seed(seed)
    num_anchors = min(max(4, n//4), len(latents))
    anchors = latents[np.random.choice(len(latents), num_anchors, replace=False)]
    path = []
    steps_per_segment = max(1, n // (len(anchors) - 1))
    for i in range(len(anchors)-1):
        for j in range(steps_per_segment):
            t = j / steps_per_segment
            t = t*t*(3-2*t)  # smoothstep
            path.append(slerp(anchors[i], anchors[i+1], t))
            if len(path) >= n:
                return np.array(path[:n])
    # Fill remaining if needed
    while len(path) < n:
        path.append(anchors[-1])
    return np.array(path[:n])

In [ ]:
# Atlas generation
@torch.no_grad()
def decode(lat):
    t = torch.from_numpy(lat).to(device, dtype=torch.float16)
    if t.dim() == 3: t = t.unsqueeze(0)
    out = vae.decode(t).sample
    return ((out/2+0.5).clamp(0,1).cpu().permute(0,2,3,1).numpy()[0]*255).astype(np.uint8)

def make_atlas(path, size, grid):
    ss = size // grid
    atlas = np.zeros((size, size, 3), dtype=np.uint8)
    for i in tqdm(range(grid*grid), desc="Decoding"):
        if i >= len(path): break
        img = np.array(Image.fromarray(decode(path[i])).resize((ss,ss), Image.LANCZOS))
        atlas[(i//grid)*ss:(i//grid+1)*ss, (i%grid)*ss:(i%grid+1)*ss] = img
    return atlas

def process(atlas, sigma=2.5):
    gray = (0.299*atlas[:,:,0]+0.587*atlas[:,:,1]+0.114*atlas[:,:,2]).astype(np.float32)/255
    norm = lambda x: (x-x.min())/(x.max()-x.min()+1e-8)
    return (np.stack([norm(gaussian_filter(gray,sigma*2)), norm(gaussian_filter(gray,sigma)), norm(gaussian_filter(gray,sigma*0.5))], -1)*255).astype(np.uint8)

def seamless(atlas, b=32):
    h,w = atlas.shape[:2]
    r = atlas.astype(np.float32)
    for i,t in enumerate(np.linspace(0,1,b)):
        r[:,i] = r[:,i]*t + r[:,w-b+i]*(1-t)
        r[:,w-b+i] = r[:,i]
        r[i,:] = r[i,:]*t + r[h-b+i,:]*(1-t)
        r[h-b+i,:] = r[i,:]
    return np.clip(r,0,255).astype(np.uint8)

In [ ]:
# Generate
out_dir = Path(CONFIG["output_dir"])
results = []

print(f"Generating {CONFIG['num_atlases']} atlases from {len(latents)} latents...")

def make_seamless_color(atlas, b=48):
    """Make color atlas seamlessly tileable"""
    h, w = atlas.shape[:2]
    r = atlas.astype(np.float32)
    for i, t in enumerate(np.linspace(0, 1, b)):
        # Horizontal seam
        r[:, i] = r[:, i] * t + r[:, w-b+i] * (1-t)
        r[:, w-b+i] = r[:, i]
        # Vertical seam
        r[i, :] = r[i, :] * t + r[h-b+i, :] * (1-t)
        r[h-b+i, :] = r[i, :]
    return np.clip(r, 0, 255).astype(np.uint8)

for idx in range(CONFIG["num_atlases"]):
    print(f"\n=== Atlas {idx+1}/{CONFIG['num_atlases']} ===")
    path = gen_path(latents, CONFIG["grid_size"]**2, seed=42+idx)
    raw = make_atlas(path, CONFIG["atlas_size"], CONFIG["grid_size"])
    raw_seamless = make_seamless_color(raw, 48)  # Seamless COLOR version
    proc = process(raw, CONFIG["blur_sigma"])
    seam = seamless(proc, 48)
    
    Image.fromarray(raw).save(out_dir/f"atlas_{idx}_art.png")
    Image.fromarray(raw_seamless).save(out_dir/f"atlas_{idx}_art_seamless.png")
    Image.fromarray(proc).save(out_dir/f"atlas_{idx}_processed.png")
    Image.fromarray(seam).save(out_dir/f"atlas_{idx}_seamless.png")
    results.append({'raw':raw, 'raw_seamless':raw_seamless, 'proc':proc, 'seam':seam})

# IMPORTANT: Use the colorful art atlas, not grayscale!
Image.fromarray(results[0]['raw_seamless']).save(out_dir/"sd_atlas.png")
print(f"\nSaved COLORFUL art atlas to {out_dir}/sd_atlas.png")

In [ ]:
# Preview
n = CONFIG["num_atlases"]
fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
if n == 1: axes = axes[np.newaxis, :]
for i, r in enumerate(results):
    for j, (k, t) in enumerate([('raw','Art'),('proc','Processed'),('seam','Seamless')]):
        axes[i,j].imshow(r[k])
        axes[i,j].set_title(f"Atlas {i+1}: {t}")
        axes[i,j].axis('off')
plt.tight_layout()
plt.show()

---
## 6. Download

In [ ]:
!cd output && zip -r ../art_atlases.zip .
from google.colab import files
files.download('art_atlases.zip')

print("""
========================================
USAGE:
1. Copy sd_atlas.png to: frontend/public/textures/
2. Set useSDTextures: true in frontend/src/main.js
3. Run: npm run dev
========================================
""")

---
## 7. Add More Images Later

```python
# Mount Drive and copy images
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/my_art/* ./images/

# Then re-run from Section 4
```